In [20]:
!pip install -U "pandas" "indic-nlp-library" "transformers[torch]" "datasets" "httpx==0.24.0" "accelerate>=0.26.0" "scikit-learn"
!git clone https://github.com/anoopkunchukuttan/indic_nlp_resources.git

fatal: destination path 'indic_nlp_resources' already exists and is not an empty directory.


In [21]:
from indicnlp import common
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
import pandas as pd
from sklearn.model_selection import train_test_split
import re
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Set up IndicNLP
common.set_resources_path("./indic_nlp_resources")
factory = IndicNormalizerFactory()
normalizer = factory.get_normalizer("ta")

# Load and clean data (assuming your CSV is uploaded to Colab as 'Tamil-News-Headlines.csv')
df = pd.read_csv('Tamil-News-Headlines.csv', encoding='utf-8')  # Or use 'utf-8-sig' if BOM issue

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\u0B80-\u0BFF\s.,!?]', '', text)
    return normalizer.normalize(text.strip()) if text.strip() else ""

df['News'] = df['News'].apply(clean_text)
df_train = df[['News', 'Authenticity']].copy()
df_train.columns = ['text', 'label']

# Split data (80/10/10)
train_df, temp_df = train_test_split(df_train, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 4180, Val: 523, Test: 523


In [24]:
model_id = "google/muril-base-cased"

# Training args
training_args = TrainingArguments(
    output_dir='./results_muril',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to=[],
    logging_strategy="epoch",
    save_total_limit=2,
    seed=42,
    fp16=True,
    push_to_hub=False  # Changed to False
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128  # Good for headlines
    )

# Prepare datasets
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_function, batched=True)
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True)).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize_function, batched=True)

train_ds = train_ds.rename_column("label", "labels").remove_columns(["text"])
val_ds = val_ds.rename_column("label", "labels").remove_columns(["text"])
test_ds = test_ds.rename_column("label", "labels").remove_columns(["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f} | F1: {test_results['eval_f1']:.4f}")

# Explicitly save the best model and tokenizer to Google Drive after training
import os

drive_model_path = '/content/gdrive/MyDrive/my_tamil_fake_news_model'
os.makedirs(drive_model_path, exist_ok=True)
model.save_pretrained(drive_model_path)
tokenizer.save_pretrained(drive_model_path)
print(f"Best model and tokenizer explicitly saved to {drive_model_path}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4180 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.381600,0.106720,0.975143,0.975881
2,0.090800,0.095000,0.977055,0.977941
3,0.040900,0.101416,0.980880,0.981618
4,0.020700,0.107358,0.977055,0.978022


Test Accuracy: 0.9809 | F1: 0.9826
Best model and tokenizer explicitly saved to /content/gdrive/MyDrive/my_tamil_fake_news_model


In [25]:
from transformers import pipeline

# Load your trained model from the new, explicitly saved local directory
classifier = pipeline("text-classification", model="/content/gdrive/MyDrive/my_tamil_fake_news_model")

# Example Tamil headline (replace with any)
headline = "பாஸ்வேர்டை பகிரும் பயனர்களிடம் கூடுதல் கட்டணம்: நெட்ஃப்ளிக்ஸ் பலே திட்டம்"  # From your sample

result = classifier(headline)[0]
label = "Real" if result['label'] == 'LABEL_0' else "Fake" # Assuming 0 is Real and 1 is Fake
confidence = result['score']
print(f"Headline: {headline}\nPrediction: {label} (Confidence: {confidence:.4f})")

The tokenizer you are loading from '/content/gdrive/MyDrive/my_tamil_fake_news_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


Headline: பாஸ்வேர்டை பகிரும் பயனர்களிடம் கூடுதல் கட்டணம்: நெட்ஃப்ளிக்ஸ் பலே திட்டம்
Prediction: Real (Confidence: 0.9940)


In [27]:
print("Evaluating on training set...")
train_results = trainer.evaluate(train_ds)
print(f"Training Accuracy: {train_results['eval_accuracy']:.4f} | F1: {train_results['eval_f1']:.4f}")

print("Evaluating on validation set...")
val_results = trainer.evaluate(val_ds)
print(f"Validation Accuracy: {val_results['eval_accuracy']:.4f} | F1: {val_results['eval_f1']:.4f}")

print(f"Test Accuracy: {test_results['eval_accuracy']:.4f} | F1: {test_results['eval_f1']:.4f}")

Evaluating on training set...


Training Accuracy: 0.9976 | F1: 0.9979
Evaluating on validation set...
Validation Accuracy: 0.9809 | F1: 0.9816
Test Accuracy: 0.9809 | F1: 0.9826


In [28]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create a text input widget for the headline
headline_input = widgets.Textarea(
    value='',
    placeholder='Type your Tamil headline here...',
    description='Headline:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='100px')
)

# Create a button widget to trigger prediction
predict_button = widgets.Button(
    description='Predict',
    disabled=False,
    button_style=''
)

# Create an output widget to display results
output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        if not headline_input.value.strip():
            print("Please enter a headline.")
            return

        # Use the already loaded classifier pipeline
        # Ensure the 'classifier' object and 'normalizer' are still in scope from previous cells
        try:
            # Clean and normalize the input headline using the previously defined normalizer
            cleaned_headline = normalizer.normalize(headline_input.value.strip())

            result = classifier(cleaned_headline)[0]
            label = "Real" if result['label'] == 'LABEL_0' else "Fake"
            confidence = result['score']

            display(HTML(f"<h3>Prediction: <span style='color: {'green' if label == 'Real' else 'red'}'>{label}</span> (Confidence: {confidence:.4f})</h3>"))
            display(HTML(f"<b>Original Headline:</b> {headline_input.value}"))
            display(HTML(f"<b>Cleaned Headline:</b> {cleaned_headline}"))
        except Exception as e:
            print(f"An error occurred during prediction: {e}")

# Attach the button click event to the function
predict_button.on_click(on_button_click)

# Display the widgets
display(headline_input, predict_button, output_area)

Textarea(value='', description='Headline:', layout=Layout(height='100px', width='auto'), placeholder='Type you…

Button(description='Predict', style=ButtonStyle())

Output()